In [1]:
import numpy as np
import matplotlib.pyplot as plt

from yasir_agdo_mt.core.forward import (
    mt1d,
    apparent_resistivity,
    phase,
)

Matplotlib is building the font cache; this may take a moment.


In [2]:
def MT1D(resistivities, thicknesses, frequency):

    mu = 4*np.pi*1e-7
    w = 2*np.pi*frequency

    rho_app = np.zeros_like(frequency)
    phase = np.zeros_like(frequency)

    for i in range(len(frequency)):

        Z = np.sqrt(1j*w[i]*mu*resistivities[-1])

        for j in range(len(resistivities)-2, -1, -1):

            rho = resistivities[j]
            h = thicknesses[j]

            dj = np.sqrt(1j*w[i]*mu/rho)
            wj = rho*dj
            ej = np.exp(-2*h*dj)

            r = (wj-Z)/(wj+Z)
            Z = wj*((1-r*ej)/(1+r*ej))

        rho_app[i] = (np.abs(Z)**2)/(mu*w[i])
        phase[i] = np.degrees(np.arctan2(Z.imag, Z.real))

    return rho_app, phase

In [3]:
freq = np.logspace(-3, 4, 56)

rho = np.array([100, 10, 500])

thick = np.array([50, 100])

In [4]:
# ============================
# Original Notebook
# ============================

rho_old, phase_old = MT1D(
    rho,
    thick,
    freq
)

# ============================
# Yasir AGDO-MT Library
# ============================

Z = mt1d(
    rho,
    thick,
    freq
)

rho_new = apparent_resistivity(
    Z,
    freq
)

phase_new = phase(
    Z
)

In [5]:
print("=" * 60)
print("FORWARD MODEL VALIDATION")
print("=" * 60)

print()

print("Apparent Resistivity")
print("---------------------")
print("Match          :", np.allclose(rho_old, rho_new))
print("Maximum Error  :", np.max(np.abs(rho_old-rho_new)))
print("Mean Error     :", np.mean(np.abs(rho_old-rho_new)))

print()

print("Phase")
print("---------------------")
print("Match          :", np.allclose(phase_old, phase_new))
print("Maximum Error  :", np.max(np.abs(phase_old-phase_new)))
print("Mean Error     :", np.mean(np.abs(phase_old-phase_new)))

FORWARD MODEL VALIDATION

Apparent Resistivity
---------------------
Match          : True
Maximum Error  : 1.0800249583553523e-12
Mean Error     : 6.534455516365207e-14

Phase
---------------------
Match          : True
Maximum Error  : 4.263256414560601e-14
Mean Error     : 6.851662094829538e-15
